# PhotoMappers Longitudinal Reasoning Analysis (2017-2025)

**Author:** NUS PhotoMapper Team  
**Date:** 2026-09-24

## Objective
Analyze year-by-year VLM cross-view geographic reasoning performance and derive reproducible summary outputs.

## Inputs
- `data/input/reasoning_yearly_2017_2025_20260918.xlsx`

## Outputs
- `data/outputs/longitudinal_reasoning_long.csv`
- `data/outputs/yearly_overall_performance.csv`
- `data/outputs/yearly_geographic_cue.csv`
- `data/outputs/yearly_reasoning_evidence.csv`
- Publication figures exported to `data/outputs/`

In [ ]:
# 1) Imports and global configuration

from pathlib import Path
import re

import numpy as np

import pandas as pd

import matplotlib as mpl

import matplotlib.pyplot as plt

from IPython.display import display

from photomapper_common import (
    detect_columns,
    detect_header_row,
    geographic_cue_from_indicator,
    reasoning_evidence_from_indicator,
)

PROJECT_DIR = Path.cwd()
INPUT_XLSX = PROJECT_DIR / 'data' / 'input' / 'reasoning_yearly_2017_2025_20260918.xlsx'
OUTPUT_DIR = PROJECT_DIR / 'data' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED_YEARS = list(range(2017, 2026))

GEOGRAPHIC_CUE_ORDER = [
    'Building and structure',
    'Road and transport',
    'Vegetation and terrain',
    'Objects and facilities',
    'Global scene',
]

REASONING_EVIDENCE_ORDER = [
    'Appearance',
    'Spatial layout',
    'Landmark cue',
]

MODEL_ORDER = ['Qwen3.6-27B', 'InternVL3-8B']

MODEL_COLORS = {
    'Qwen3.6-27B': '#2166AC',
    'InternVL3-8B': '#B2182B',
}

EVIDENCE_COLORS = {
    'Appearance': '#9ECAE1',
    'Spatial layout': '#3182BD',
    'Landmark cue': '#08519C',
}

CUE_COLORS = {
    'Building and structure': '#2166AC',
    'Road and transport': '#67A9CF',
    'Vegetation and terrain': '#A6D96A',
    'Objects and facilities': '#1B7837',
    'Global scene': '#00441B',
}

mpl.rcParams.update({
    'font.family': 'Arial',
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'axes.linewidth': 0.8,
})

if not INPUT_XLSX.exists():
    raise FileNotFoundError(f'Input workbook not found: {INPUT_XLSX}')

print(f'Project directory: {PROJECT_DIR}')
print(f'Input workbook: {INPUT_XLSX}')
print(f'Output directory: {OUTPUT_DIR}')

In [ ]:
# 2) Workbook inspection and schema detection

excel = pd.ExcelFile(INPUT_XLSX)
sheet_names = excel.sheet_names

inspection_rows = []
parsed_indicator_tables = []
parsed_summary_tables = []

for sheet in sheet_names:
    raw = pd.read_excel(INPUT_XLSX, sheet_name=sheet, header=None)
    header_idx = detect_header_row(raw)
    header_vals = raw.iloc[header_idx].tolist()
    col_map = detect_columns(header_vals)

    body = raw.iloc[header_idx + 1:].copy()
    body.columns = header_vals
    body = body.dropna(how='all')

    body[col_map['no_col']] = pd.to_numeric(body[col_map['no_col']], errors='coerce')
    indicator_rows = body[body[col_map['no_col']].between(1, 30, inclusive='both')].copy()

    for metric_col in [
        col_map['Qwen3.6-27B_kappa'],
        col_map['Qwen3.6-27B_agr'],
        col_map['InternVL3-8B_kappa'],
        col_map['InternVL3-8B_agr'],
    ]:
        indicator_rows[metric_col] = pd.to_numeric(indicator_rows[metric_col], errors='coerce')

    indicator_rows['Year'] = int(sheet) if str(sheet).isdigit() else np.nan
    indicator_rows['Indicator_No'] = indicator_rows[col_map['no_col']].astype(int)
    indicator_rows['Indicator'] = indicator_rows[col_map['indicator_col']].astype(str).str.strip()
    indicator_rows['Geographic_Cue'] = indicator_rows['Indicator_No'].map(geographic_cue_from_indicator)
    indicator_rows['Reasoning_Evidence'] = indicator_rows['Indicator_No'].map(reasoning_evidence_from_indicator)

    parsed_indicator_tables.append(pd.DataFrame({
        'Year': indicator_rows['Year'],
        'Indicator_No': indicator_rows['Indicator_No'],
        'Indicator': indicator_rows['Indicator'],
        'Geographic_Cue': indicator_rows['Geographic_Cue'],
        'Reasoning_Evidence': indicator_rows['Reasoning_Evidence'],
        'Qwen3.6-27B_kappa_w': indicator_rows[col_map['Qwen3.6-27B_kappa']],
        'Qwen3.6-27B_agreement': indicator_rows[col_map['Qwen3.6-27B_agr']],
        'InternVL3-8B_kappa_w': indicator_rows[col_map['InternVL3-8B_kappa']],
        'InternVL3-8B_agreement': indicator_rows[col_map['InternVL3-8B_agr']],
    }))

    summary_rows = body[body[col_map['no_col']].isna() & body[col_map['indicator_col']].notna()].copy()
    for metric_col in [
        col_map['Qwen3.6-27B_kappa'],
        col_map['Qwen3.6-27B_agr'],
        col_map['InternVL3-8B_kappa'],
        col_map['InternVL3-8B_agr'],
    ]:
        summary_rows[metric_col] = pd.to_numeric(summary_rows[metric_col], errors='coerce')

    parsed_summary_tables.append(pd.DataFrame({
        'Year': int(sheet) if str(sheet).isdigit() else np.nan,
        'Summary_Label': summary_rows[col_map['indicator_col']].astype(str).str.strip(),
        'Qwen3.6-27B_kappa_w': summary_rows[col_map['Qwen3.6-27B_kappa']],
        'Qwen3.6-27B_agreement': summary_rows[col_map['Qwen3.6-27B_agr']],
        'InternVL3-8B_kappa_w': summary_rows[col_map['InternVL3-8B_kappa']],
        'InternVL3-8B_agreement': summary_rows[col_map['InternVL3-8B_agr']],
    }))

    inspection_rows.append({
        'Sheet': sheet,
        'Detected_Year': int(sheet) if str(sheet).isdigit() else np.nan,
        'Rows': raw.shape[0],
        'Columns': raw.shape[1],
        'Header_Row_Index': header_idx,
        'Indicator_Rows_Detected': int(indicator_rows.shape[0]),
        'Summary_Rows_Detected': int(summary_rows.shape[0]),
        'No_Column': col_map['no_col'],
        'Indicator_Column': col_map['indicator_col'],
        'Qwen_kappa_column': col_map['Qwen3.6-27B_kappa'],
        'Qwen_agreement_column': col_map['Qwen3.6-27B_agr'],
        'Intern_kappa_column': col_map['InternVL3-8B_kappa'],
        'Intern_agreement_column': col_map['InternVL3-8B_agr'],
    })

inspection_df = pd.DataFrame(inspection_rows)
indicator_wide_df = pd.concat(parsed_indicator_tables, ignore_index=True)
summary_df = pd.concat(parsed_summary_tables, ignore_index=True)

print('Detected worksheets:')
print(sheet_names)
print('\nWorkbook structure inspection:')
display(inspection_df)

print('\nDetected data types (indicator-level table):')
display(indicator_wide_df.dtypes.to_frame('dtype'))

expected_sheet_set = set(str(y) for y in EXPECTED_YEARS)
actual_sheet_set = set(str(s) for s in sheet_names)
print('\nYear-sheet coverage check (2017-2025):', expected_sheet_set == actual_sheet_set)
print('Missing expected sheets:', sorted(expected_sheet_set - actual_sheet_set))
print('Unexpected extra sheets:', sorted(actual_sheet_set - expected_sheet_set))

## Detected Structure Notes

The workbook uses a repeated year-sheet format (2017-2025), each with:
- a title row,
- a detected header row containing indicator/model metric names,
- 30 indicator-level rows (`No.` = 1..30),
- multiple provided summary-average rows (cue-level, evidence-level, and overall).

No explicit Year/Geographic Cue/Reasoning Evidence columns exist in raw rows.
- Year is taken from worksheet name.
- Geographic Cue is assigned by indicator-number blocks (1-6, 7-12, 13-18, 19-24, 25-30).
- Reasoning Evidence is assigned by within-block position (1-2 Appearance, 3-4 Spatial layout, 5-6 Landmark cue).

These assignments are validated against provided summary-average rows using numerical tolerance.

In [ ]:
# 3) Build cleaned long-format dataset

long_df = indicator_wide_df.melt(

    id_vars=['Year', 'Indicator_No', 'Indicator', 'Geographic_Cue', 'Reasoning_Evidence'],

    value_vars=[

        'Qwen3.6-27B_kappa_w',

        'Qwen3.6-27B_agreement',

        'InternVL3-8B_kappa_w',

        'InternVL3-8B_agreement',

    ],

    var_name='Model_Metric',

    value_name='Value',

)



long_df['Model'] = np.where(long_df['Model_Metric'].str.startswith('Qwen3.6-27B'), 'Qwen3.6-27B', 'InternVL3-8B')

long_df['Metric'] = np.where(long_df['Model_Metric'].str.contains('kappa_w'), 'kappa_w', 'agreement_rate')



long_df = long_df.drop(columns=['Model_Metric'])

long_df = long_df.pivot_table(

    index=['Year', 'Indicator_No', 'Indicator', 'Geographic_Cue', 'Reasoning_Evidence', 'Model'],

    columns='Metric',

    values='Value',

    aggfunc='first'

).reset_index()



long_df.columns.name = None

long_df = long_df.sort_values(['Year', 'Model', 'Indicator_No']).reset_index(drop=True)



for c in ['kappa_w', 'agreement_rate']:

    long_df[c] = pd.to_numeric(long_df[c], errors='coerce')



long_csv_path = OUTPUT_DIR / 'longitudinal_reasoning_long.csv'

long_df.to_csv(long_csv_path, index=False)



print(f'Saved: {long_csv_path}')

display(long_df.head(12))

In [ ]:
# 4) Validation checks before figure generation

def _summary_label_norm(s):

    s = str(s).strip().lower()

    s = re.sub(r'\s+', ' ', s)

    return s



# Check 1: confirm all years 2017-2025 are present in parsed indicator-level data

years_present = sorted(long_df['Year'].dropna().astype(int).unique())

missing_years = sorted(set(EXPECTED_YEARS) - set(years_present))



# Check 2: confirm 30 indicators for every year-model combination

counts_by_year_model = long_df.groupby(['Year', 'Model'])['Indicator_No'].nunique().reset_index(name='n_indicators')

invalid_indicator_counts = counts_by_year_model[counts_by_year_model['n_indicators'] != 30].copy()



# Check 3-4: identify/report missing values by year-model

missing_tbl = long_df.groupby(['Year', 'Model']).agg(

    missing_kappa=('kappa_w', lambda s: int(s.isna().sum())),

    missing_agreement=('agreement_rate', lambda s: int(s.isna().sum())),

).reset_index()

missing_tbl['missing_total'] = missing_tbl['missing_kappa'] + missing_tbl['missing_agreement']



# Check 5: verify calculated group means against workbook summary rows (small tolerance)

cue_means = long_df.groupby(['Year', 'Model', 'Geographic_Cue'], as_index=False).agg(

    mean_kappa_w=('kappa_w', 'mean'),

    mean_agreement=('agreement_rate', 'mean'),

)



evidence_means = long_df.groupby(['Year', 'Model', 'Reasoning_Evidence'], as_index=False).agg(

    mean_kappa_w=('kappa_w', 'mean'),

    mean_agreement=('agreement_rate', 'mean'),

)



overall_means = long_df.groupby(['Year', 'Model'], as_index=False).agg(

    mean_kappa_w=('kappa_w', 'mean'),

    mean_agreement=('agreement_rate', 'mean'),

)



label_to_cue = {

    'building and structure average': 'Building and structure',

    'road and transport average': 'Road and transport',

    'vegetation and terrain average': 'Vegetation and terrain',

    'objects and facilities average': 'Objects and facilities',

    'global scene average': 'Global scene',

}

label_to_evidence = {

    'appearance dimension average': 'Appearance',

    'spatial layout dimension average': 'Spatial layout',

    'landmark cue dimension average': 'Landmark cue',

}

overall_label = 'overall indicator-level average'



summary_long = []

for _, row in summary_df.iterrows():

    y = int(row['Year'])

    label = _summary_label_norm(row['Summary_Label'])

    for model in MODEL_ORDER:

        summary_long.append({

            'Year': y,

            'Summary_Label': label,

            'Model': model,

            'summary_kappa_w': row[f'{model}_kappa_w'],

            'summary_agreement': row[f'{model}_agreement'],

        })

summary_long = pd.DataFrame(summary_long)



calc_compare_rows = []

for _, r in summary_long.iterrows():

    y, label, model = int(r['Year']), r['Summary_Label'], r['Model']

    if label in label_to_cue:

        cue = label_to_cue[label]

        c = cue_means[(cue_means['Year'] == y) & (cue_means['Model'] == model) & (cue_means['Geographic_Cue'] == cue)]

    elif label in label_to_evidence:

        ev = label_to_evidence[label]

        c = evidence_means[(evidence_means['Year'] == y) & (evidence_means['Model'] == model) & (evidence_means['Reasoning_Evidence'] == ev)]

    elif label == overall_label:

        c = overall_means[(overall_means['Year'] == y) & (overall_means['Model'] == model)]

    else:

        continue



    if len(c) == 1:

        calc_compare_rows.append({

            'Year': y,

            'Model': model,

            'Summary_Label': label,

            'summary_kappa_w': r['summary_kappa_w'],

            'calc_kappa_w': float(c['mean_kappa_w'].iloc[0]),

            'delta_kappa_w': float(c['mean_kappa_w'].iloc[0] - r['summary_kappa_w']),

            'summary_agreement': r['summary_agreement'],

            'calc_agreement': float(c['mean_agreement'].iloc[0]),

            'delta_agreement': float(c['mean_agreement'].iloc[0] - r['summary_agreement']),

        })



validation_compare = pd.DataFrame(calc_compare_rows)

tolerance = 1e-6

validation_compare['kappa_within_tol'] = validation_compare['delta_kappa_w'].abs() <= tolerance

validation_compare['agreement_within_tol'] = validation_compare['delta_agreement'].abs() <= tolerance



# Check 6: verify yearly overall means are direct means from 30 indicators

overall_direct_check = long_df.groupby(['Year', 'Model']).agg(

    n_indicator_rows=('Indicator_No', 'nunique'),

    mean_kappa_w_direct=('kappa_w', 'mean'),

    mean_agreement_direct=('agreement_rate', 'mean'),

).reset_index()



# Check 7: ensure no manual rounding in calculations

# We preserve raw numeric values as read from workbook and only format at display/export if needed.

non_null_vals = long_df[['kappa_w', 'agreement_rate']].stack().dropna()

precision_signal = (non_null_vals * 1000000).round(0).astype(int)

has_sub_1e3_precision = bool(((precision_signal % 1000) != 0).any())



# Missing-value handling policy: do not silently drop

if missing_tbl['missing_total'].sum() > 0:

    print('Missing values detected. Means below use pandas mean(), which excludes NaN by definition.')

    print('Missingness is explicitly reported and retained in validation tables.')

else:

    print('No missing metric values detected across expected year-model combinations.')



print('\nValidation summary:')

print(f'Years present: {years_present}')

print(f'Missing expected years: {missing_years}')

print(f'Invalid year-model indicator counts (!=30): {len(invalid_indicator_counts)}')

print(f'Rows compared to workbook-provided summaries: {len(validation_compare)}')

print(f'Max |delta kappa_w|: {validation_compare["delta_kappa_w"].abs().max():.10f}')

print(f'Max |delta agreement|: {validation_compare["delta_agreement"].abs().max():.10f}')

print(f'Evidence that high precision values are used (not coarse rounded only): {has_sub_1e3_precision}')



print('\nMissingness by year and model:')

display(missing_tbl)



print('Validation comparison table (first 20 rows):')

display(validation_compare.head(20))



if missing_years:

    raise ValueError(f'Missing expected years in parsed data: {missing_years}')

if not invalid_indicator_counts.empty:

    raise ValueError('Some year-model combinations do not have exactly 30 indicators.')

In [ ]:
# 5) Export required analytical tables

yearly_overall = overall_means.copy().rename(columns={

    'mean_kappa_w': 'Mean kappa_w',

    'mean_agreement': 'Mean agreement',

})



yearly_geographic_cue = cue_means.copy().rename(columns={

    'Geographic_Cue': 'Geographic Cue',

    'mean_kappa_w': 'Mean kappa_w',

    'mean_agreement': 'Mean agreement',

})



yearly_reasoning_evidence = evidence_means.copy().rename(columns={

    'Reasoning_Evidence': 'Reasoning Evidence',

    'mean_kappa_w': 'Mean kappa_w',

    'mean_agreement': 'Mean agreement',

})



yearly_overall_csv = OUTPUT_DIR / 'yearly_overall_performance.csv'

yearly_geographic_csv = OUTPUT_DIR / 'yearly_geographic_cue.csv'

yearly_evidence_csv = OUTPUT_DIR / 'yearly_reasoning_evidence.csv'



yearly_overall.to_csv(yearly_overall_csv, index=False)

yearly_geographic_cue.to_csv(yearly_geographic_csv, index=False)

yearly_reasoning_evidence.to_csv(yearly_evidence_csv, index=False)



print(f'Saved: {yearly_overall_csv}')

print(f'Saved: {yearly_geographic_csv}')

print(f'Saved: {yearly_evidence_csv}')



display(yearly_overall.head())

In [ ]:
# 6) Plot helpers and Figure 1-3 generation



def style_axes(ax):



    ax.spines['top'].set_visible(False)



    ax.spines['right'].set_visible(False)



    ax.grid(axis='y', linestyle='-', linewidth=0.4, alpha=0.25)



    ax.grid(axis='x', visible=False)







def save_fig(fig, png_path):



    fig.savefig(png_path, dpi=600, transparent=True, bbox_inches='tight')



def save_legend_png(handles, labels, title, output_png, ncol=3):



    fig_leg, ax_leg = plt.subplots(figsize=(8.0, 1.8))



    ax_leg.axis('off')



    fig_leg.legend(



        handles,



        labels,



        title=title,



        loc='center',



        frameon=False,



        ncol=ncol,



    )



    fig_leg.savefig(output_png, dpi=600, transparent=True, bbox_inches='tight')



    plt.close(fig_leg)







# Explicit marker mapping for clearer curve distinction.



MODEL_MARKERS = {



    'Qwen3.6-27B': 'o',      # circle



    'InternVL3-8B': '^',     # triangle



}







EVIDENCE_MARKERS = {



    'Appearance': 'o',       # circle



    'Spatial layout': '^',   # triangle



    'Landmark cue': 's',     # square/rectangular



}







CUE_MARKERS = {



    'Building and structure': 'o',



    'Road and transport': '^',



    'Vegetation and terrain': 's',



    'Objects and facilities': 'D',



    'Global scene': 'v',



}








AGREEMENT_EVIDENCE_COLORS = {
    'Appearance': '#FCAE91',
    'Spatial layout': '#FB6A4A',
    'Landmark cue': '#CB181D',
}

AGREEMENT_CUE_COLORS = {
    'Building and structure': '#FEE5D9',
    'Road and transport': '#FCBBA1',
    'Vegetation and terrain': '#FC9272',
    'Objects and facilities': '#FB6A4A',
    'Global scene': '#CB181D',
}

# Figure 1: Overall longitudinal performance



fig1, axes = plt.subplots(1, 2, figsize=(10.8, 4.2), sharex=True)







for model in MODEL_ORDER:



    sub = yearly_overall[yearly_overall['Model'] == model].sort_values('Year')



    axes[0].plot(



        sub['Year'],



        sub['Mean kappa_w'],



        marker=MODEL_MARKERS[model],



        markersize=4,



        linewidth=2.0,



        color=MODEL_COLORS[model],



        label=model,



    )



    axes[1].plot(



        sub['Year'],



        sub['Mean agreement'] * 100.0,



        marker=MODEL_MARKERS[model],



        markersize=4,



        linewidth=2.0,



        color=MODEL_COLORS[model],



        label=model,



    )







for ax in axes:



    ax.set_xlabel('')



    ax.set_ylabel('')



    ax.set_xticks(EXPECTED_YEARS)



    style_axes(ax)







axes[0].set_title('(a) Mean $\\kappa_w$ by year')



axes[1].set_title('(b) Mean agreement by year')



handles1, labels1 = axes[1].get_legend_handles_labels()
for _ax in axes:
    leg = _ax.get_legend()
    if leg is not None:
        leg.remove()
fig1.tight_layout()
legend1_png = OUTPUT_DIR / 'Legend_Fig1_Overall_Model.png'
save_legend_png(handles1, labels1, 'Model', legend1_png, ncol=2)







fig1_png = OUTPUT_DIR / 'Fig1_Longitudinal_Overall_Reasoning.png'



save_fig(fig1, fig1_png)



plt.show()







# Figure 2: Reasoning evidence breakdown (kappa_w)



fig2, axes = plt.subplots(1, 2, figsize=(10.8, 4.2), sharex=True, sharey=True)







ymin2 = evidence_means['mean_kappa_w'].min()



ymax2 = evidence_means['mean_kappa_w'].max()



pad2 = (ymax2 - ymin2) * 0.08 if ymax2 > ymin2 else 0.02







for i, model in enumerate(MODEL_ORDER):



    ax = axes[i]



    d = evidence_means[evidence_means['Model'] == model]



    for ev in REASONING_EVIDENCE_ORDER:



        s = d[d['Reasoning_Evidence'] == ev].sort_values('Year')



        ax.plot(



            s['Year'],



            s['mean_kappa_w'],



            marker=EVIDENCE_MARKERS[ev],



            markersize=4,



            linewidth=2.0,



            color=EVIDENCE_COLORS[ev],



            label=ev,



        )



    ax.set_title('(a) Qwen3.6-27B' if i == 0 else '(b) InternVL3-8B')



    ax.set_xlabel('')



    ax.set_ylabel('')



    ax.set_xticks(EXPECTED_YEARS)



    style_axes(ax)







axes[0].set_ylim(ymin2 - pad2, ymax2 + pad2)



handles2, labels2 = axes[1].get_legend_handles_labels()
for _ax in axes:
    leg = _ax.get_legend()
    if leg is not None:
        leg.remove()
fig2.tight_layout()
legend2_png = OUTPUT_DIR / 'Legend_Fig2_ReasoningEvidence_Kappa.png'
save_legend_png(handles2, labels2, 'Reasoning evidence', legend2_png, ncol=3)







fig2_png = OUTPUT_DIR / 'Fig2_Longitudinal_Reasoning_Evidence.png'



save_fig(fig2, fig2_png)



plt.show()







# Figure 3: Geographic cue breakdown (kappa_w)



fig3, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), sharex=True, sharey=True)







ymin3 = cue_means['mean_kappa_w'].min()



ymax3 = cue_means['mean_kappa_w'].max()



pad3 = (ymax3 - ymin3) * 0.08 if ymax3 > ymin3 else 0.02







for i, model in enumerate(MODEL_ORDER):



    ax = axes[i]



    d = cue_means[cue_means['Model'] == model]



    for cue in GEOGRAPHIC_CUE_ORDER:



        s = d[d['Geographic_Cue'] == cue].sort_values('Year')



        ax.plot(



            s['Year'],



            s['mean_kappa_w'],



            marker=CUE_MARKERS[cue],



            markersize=4,



            linewidth=1.9,



            color=CUE_COLORS[cue],



            label=cue,



        )



    ax.set_title('(a) Qwen3.6-27B' if i == 0 else '(b) InternVL3-8B')



    ax.set_xlabel('')



    ax.set_ylabel('')



    ax.set_xticks(EXPECTED_YEARS)



    style_axes(ax)







axes[0].set_ylim(ymin3 - pad3, ymax3 + pad3)



handles3, labels3 = axes[1].get_legend_handles_labels()
for _ax in axes:
    leg = _ax.get_legend()
    if leg is not None:
        leg.remove()
fig3.tight_layout()
legend3_png = OUTPUT_DIR / 'Legend_Fig3_GeographicCue_Kappa.png'
save_legend_png(handles3, labels3, 'Geographic cue', legend3_png, ncol=3)







fig3_png = OUTPUT_DIR / 'Fig3_Longitudinal_Geographic_Cues.png'



save_fig(fig3, fig3_png)



plt.show()







print(f'Saved: {fig1_png}')



print(f'Saved: {fig2_png}')



print(f'Saved: {fig3_png}')


# Figure 4: Reasoning evidence breakdown (mean agreement)
fig4, axes = plt.subplots(1, 2, figsize=(10.8, 4.2), sharex=True, sharey=True)

ymin4 = evidence_means['mean_agreement'].min() * 100.0
ymax4 = evidence_means['mean_agreement'].max() * 100.0
pad4 = (ymax4 - ymin4) * 0.08 if ymax4 > ymin4 else 1.0

for i, model in enumerate(MODEL_ORDER):
    ax = axes[i]
    d = evidence_means[evidence_means['Model'] == model]
    for ev in REASONING_EVIDENCE_ORDER:
        s = d[d['Reasoning_Evidence'] == ev].sort_values('Year')
        ax.plot(
            s['Year'],
            s['mean_agreement'] * 100.0,
            marker=EVIDENCE_MARKERS[ev],
            markersize=4,
            linewidth=2.0,
            color=AGREEMENT_EVIDENCE_COLORS[ev],
            label=ev,
        )
    ax.set_title('(a) Qwen3.6-27B' if i == 0 else '(b) InternVL3-8B')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticks(EXPECTED_YEARS)
    style_axes(ax)

axes[0].set_ylim(ymin4 - pad4, ymax4 + pad4)
handles4, labels4 = axes[1].get_legend_handles_labels()
for _ax in axes:
    leg = _ax.get_legend()
    if leg is not None:
        leg.remove()
fig4.tight_layout()
legend4_png = OUTPUT_DIR / 'Legend_Fig4_ReasoningEvidence_Agreement.png'
save_legend_png(handles4, labels4, 'Reasoning evidence', legend4_png, ncol=3)

fig4_png = OUTPUT_DIR / 'Fig4_Longitudinal_Reasoning_Evidence_Agreement.png'
save_fig(fig4, fig4_png)
plt.show()

# Figure 5: Geographic cue breakdown (mean agreement)
fig5, axes = plt.subplots(1, 2, figsize=(11.5, 4.2), sharex=True, sharey=True)

ymin5 = cue_means['mean_agreement'].min() * 100.0
ymax5 = cue_means['mean_agreement'].max() * 100.0
pad5 = (ymax5 - ymin5) * 0.08 if ymax5 > ymin5 else 1.0

for i, model in enumerate(MODEL_ORDER):
    ax = axes[i]
    d = cue_means[cue_means['Model'] == model]
    for cue in GEOGRAPHIC_CUE_ORDER:
        s = d[d['Geographic_Cue'] == cue].sort_values('Year')
        ax.plot(
            s['Year'],
            s['mean_agreement'] * 100.0,
            marker=CUE_MARKERS[cue],
            markersize=4,
            linewidth=1.9,
            color=AGREEMENT_CUE_COLORS[cue],
            label=cue,
        )
    ax.set_title('(a) Qwen3.6-27B' if i == 0 else '(b) InternVL3-8B')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticks(EXPECTED_YEARS)
    style_axes(ax)

axes[0].set_ylim(ymin5 - pad5, ymax5 + pad5)
handles5, labels5 = axes[1].get_legend_handles_labels()
for _ax in axes:
    leg = _ax.get_legend()
    if leg is not None:
        leg.remove()
fig5.tight_layout()
legend5_png = OUTPUT_DIR / 'Legend_Fig5_GeographicCue_Agreement.png'
save_legend_png(handles5, labels5, 'Geographic cue', legend5_png, ncol=3)

fig5_png = OUTPUT_DIR / 'Fig5_Longitudinal_Geographic_Cues_Agreement.png'
save_fig(fig5, fig5_png)
plt.show()

print(f'Saved: {fig4_png}')
print(f'Saved: {fig5_png}')
print(f'Saved: {legend1_png}')
print(f'Saved: {legend2_png}')
print(f'Saved: {legend3_png}')
print(f'Saved: {legend4_png}')
print(f'Saved: {legend5_png}')


In [ ]:
# 7) Supplementary heatmaps (Figure S1)



def plot_indicator_heatmap(model_name, out_png, vmin, vmax, value_col='kappa_w', title_metric='kappa_w', cmap='Blues'):



    d = long_df[long_df['Model'] == model_name].copy()



    pivot = d.pivot_table(index=['Indicator_No', 'Indicator', 'Geographic_Cue'], columns='Year', values=value_col, aggfunc='first')



    pivot = pivot.sort_index(level=0)







    years = EXPECTED_YEARS



    pivot = pivot.reindex(columns=years)







    fig, ax = plt.subplots(figsize=(9.5, 8.5))



    mat = pivot.values



    im = ax.imshow(mat, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax)







    ax.set_xticks(np.arange(len(years)))



    ax.set_xticklabels(years, rotation=45, ha='right')



    ax.set_yticks(np.arange(len(pivot.index)))



    ax.set_yticklabels([f"{i:02d}" for i, _, _ in pivot.index], fontsize=10)







    for sep in [5.5, 11.5, 17.5, 23.5]:



        ax.axhline(sep, color='white', linewidth=1.2, alpha=0.9)







    # Heatmap title removed per request.



    ax.set_xlabel('')



    ax.set_ylabel('')







    cbar = fig.colorbar(im, ax=ax, fraction=0.02, pad=0.02)



    cbar.set_label(title_metric)







    fig.savefig(out_png, dpi=600, transparent=True, bbox_inches='tight')



    plt.show()







vmin_h = long_df['kappa_w'].min()



vmax_h = long_df['kappa_w'].max()







figs1_q_png = OUTPUT_DIR / 'FigS1_Indicator_Heatmap_Qwen36.png'



figs1_i_png = OUTPUT_DIR / 'FigS1_Indicator_Heatmap_InternVL.png'







plot_indicator_heatmap('Qwen3.6-27B', figs1_q_png, vmin_h, vmax_h, value_col='kappa_w', title_metric='kappa_w')



plot_indicator_heatmap('InternVL3-8B', figs1_i_png, vmin_h, vmax_h, value_col='kappa_w', title_metric='kappa_w')







print(f'Saved: {figs1_q_png}')



print(f'Saved: {figs1_i_png}')


# Agreement-rate heatmaps with shared color scale across models
vmin_a = long_df['agreement_rate'].min() * 100.0
vmax_a = long_df['agreement_rate'].max() * 100.0

figs2_q_png = OUTPUT_DIR / 'FigS2_Indicator_Heatmap_Agreement_Qwen36.png'
figs2_i_png = OUTPUT_DIR / 'FigS2_Indicator_Heatmap_Agreement_InternVL.png'

long_df['_agreement_pct'] = long_df['agreement_rate'] * 100.0
plot_indicator_heatmap('Qwen3.6-27B', figs2_q_png, vmin_a, vmax_a, value_col='_agreement_pct', title_metric='agreement_rate (%)', cmap='Reds')
plot_indicator_heatmap('InternVL3-8B', figs2_i_png, vmin_a, vmax_a, value_col='_agreement_pct', title_metric='agreement_rate (%)', cmap='Reds')
long_df = long_df.drop(columns=['_agreement_pct'])

print(f'Saved: {figs2_q_png}')
print(f'Saved: {figs2_i_png}')


In [ ]:
# 8) Final reproducibility report and concise interpretation



from IPython.display import Markdown, display







exported_files = [



    long_csv_path,



    yearly_overall_csv,



    yearly_geographic_csv,



    yearly_evidence_csv,



    fig1_png,



    fig2_png,



    fig3_png,
    fig4_png,
    fig5_png,
    legend1_png,
    legend2_png,
    legend3_png,
    legend4_png,
    legend5_png,



    figs1_q_png,



    figs1_i_png,
    figs2_q_png,
    figs2_i_png,



]







n_years = long_df['Year'].nunique()



ind_per_year = long_df.groupby('Year')['Indicator_No'].nunique().sort_index()



n_missing_total = int(long_df['kappa_w'].isna().sum() + long_df['agreement_rate'].isna().sum())







print('=== FINAL SUMMARY ===')



print(f'1) Number of years analyzed: {n_years}')



print('2) Number of indicators per year:')



print(ind_per_year.to_string())



print(f'3) Number of missing observations (kappa_w + agreement): {n_missing_total}')







print('4) Overall mean kappa_w by model and year:')



display(yearly_overall[['Year', 'Model', 'Mean kappa_w']].sort_values(['Model', 'Year']))







print('5) Overall mean agreement by model and year:')



display(yearly_overall[['Year', 'Model', 'Mean agreement']].sort_values(['Model', 'Year']))







print('6) Exported files:')



for fp in exported_files:



    print('-', fp)







trend_lines = []



for model in MODEL_ORDER:



    sub = yearly_overall[yearly_overall['Model'] == model].sort_values('Year')



    dk = sub['Mean kappa_w'].iloc[-1] - sub['Mean kappa_w'].iloc[0]



    da = sub['Mean agreement'].iloc[-1] - sub['Mean agreement'].iloc[0]



    trend_lines.append(f'- {model}: overall kappa_w changed by {dk:+.3f}; agreement changed by {da:+.3f} (fractional scale).')







md = (



    '### Concise Interpretation (Descriptive Only)\n\n'



    'The 2017-2025 curves show temporal variation in both agreement and kappa-based reasoning performance for both VLMs. '



    'Patterns in Figures 2 and 3 indicate that changes are not uniform across reasoning evidence dimensions or geographic cue groups, '



    'with some categories exhibiting stronger year-to-year fluctuation than others.\n\n'



    + '\n'.join(trend_lines) + '\n\n'



    'These trajectories are descriptive longitudinal patterns from indicator-level summaries and should not be interpreted as causal effects.'



)



display(Markdown(md))
